In [1]:
from Import_loader import *
import torch

torch device: cpu
Switched off Tx:  2
Switched off Rx:  10


In [8]:
# Experiment
utils.load_config("sphere_pos.json")
config["ref_position"] = 1
config["device"] = "cpu" # TODO: disable this again
config["use_sms"] = True # TODO: disable this again

In [9]:
# Setup
params = setup_utils.setup_experiment(config)
scene = params['scene']
ref = params['ref_msr']
update_fn = params['update_fn']
gt = setup_utils.get_gt()

print(gt)

Loaded Reference: calibrated64/sphere_pos_1_foam_64f_330_mm
-0.10775


In [10]:
hyper_params = {
    'nsamples': 1,
    'sigma': 0.0125,
    'initial_translation': [0.0],
    'gt_translation': [gt],
    'learning_rate': 2e-2,
    'epochs': 50,
    'sigma_annealing': False,
    'anneal_const_first': 20,
    'anneal_const_last': 0,
    'anneal_sigma_min': 0.001,
}

ctx_args = {
    'scene': scene, 'init_pos': [0.0], 'gt_msr': ref,
    'update_fn': update_fn, 'loss_fn': losses.torch_fft_l2,
    'sampler': 'importance', 'antithetic': True, 'nsamples': hyper_params['nsamples'],
    'sigma': hyper_params['sigma'], 'device': config['device']
}

In [11]:
# Set up initial and gt translation
initial_translation = torch.tensor(hyper_params['initial_translation'], requires_grad=True, device=config['device'])
gt_translation = torch.tensor(hyper_params['gt_translation'], device=config['device'])

# Set up optimizer:
optim = torch.optim.Adam([initial_translation], lr=hyper_params['learning_rate'])

In [12]:
theta_optim = opt.run_optimization(
    hparams=hyper_params,
    optim=optim,
    theta=initial_translation,
    gt_theta=gt_translation,
    ctx_args=ctx_args,
    schedule_fn=opt.run_scheduler_step,
    update_fn=update_fn,
)

shape =  (43648, 1)
Running 50 epochs with 1 samples and sigma=0.0125
Iter 0/50, ParamLoss: [0.10775], L2HammLoss: 8184.45410156
shape =  (43648, 1)
shape =  (43648, 1)
shape =  (43648, 1)
theta optim: tensor([-0.0200]), min loss: 8145.3427734375
Iter 1/50, ParamLoss: [0.08775], L2HammLoss: 8145.34277344 - Time: 11.2257
shape =  (43648, 1)
shape =  (43648, 1)
shape =  (43648, 1)
theta optim: tensor([-0.0356]), min loss: 8060.0400390625
Iter 2/50, ParamLoss: [0.072137], L2HammLoss: 8060.04003906 - Time: 10.3931
shape =  (43648, 1)
shape =  (43648, 1)
shape =  (43648, 1)
theta optim: tensor([-0.0529]), min loss: 8025.0146484375
Iter 3/50, ParamLoss: [0.054867], L2HammLoss: 8025.01464844 - Time: 11.9910
shape =  (43648, 1)
shape =  (43648, 1)
shape =  (43648, 1)
theta optim: tensor([-0.0667]), min loss: 7988.82177734375
Iter 4/50, ParamLoss: [0.041097], L2HammLoss: 7988.82177734 - Time: 11.5834
shape =  (43648, 1)
shape =  (43648, 1)
shape =  (43648, 1)
Iter 5/50, ParamLoss: [0.032873], L

RuntimeError: drjit.custom(<drjit.interop.WrapADOp>): error while performing a custom differentiable operation. (see above).

In [ ]:
print(theta_optim)

tensor([-0.0648])
